In [ ]:
import json
from pathlib import Path

import torch
import torch.nn as nn


# Keep architecture identical to the training notebook so state_dict loading matches.
class CNNBinary(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, kernel_size=3),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 30 * 30, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(64, 1),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


# Use GPU when available.
if torch.cuda.is_available():
    device = torch.device("cuda")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available() and torch.backends.mps.is_built():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

root = Path(".")
walls_model_path = root / "model_walls.pth"
decks_model_path = root / "model_decks.pth"
metadata_path = root / "model_metadata.json"

for p in [walls_model_path, decks_model_path, metadata_path]:
    if not p.exists():
        raise FileNotFoundError(f"Missing required file: {p.resolve()}")

model_walls = CNNBinary().to(device)
model_decks = CNNBinary().to(device)

model_walls.load_state_dict(torch.load(walls_model_path, map_location=device))
model_decks.load_state_dict(torch.load(decks_model_path, map_location=device))

model_walls.eval()
model_decks.eval()

with metadata_path.open("r", encoding="utf-8") as f:
    metadata = json.load(f)

best_threshold_walls = metadata.get("best_threshold_walls", 0.5)
best_threshold_decks = metadata.get("best_threshold_decks", 0.5)

print("Models and metadata loaded successfully.")
print(f"Device: {device}")
print(f"Walls threshold: {best_threshold_walls}")
print(f"Decks threshold: {best_threshold_decks}")